# Prototype: Analyse auf bereinigten Daten

Dieses Notebook testet die Analysefragen auf einem 1%-Sample der bereinigten Parking-Violations-Daten (`parking_violations_cleaned_v5`).

## Ziel

- bereinigte Parquet-Daten aus HDFS laden
- 1%-Sample erstellen für schnelle Exploration
- Fragestellung 1 testen: häufigste Violation Codes nach Fiskaljahr
- Fragestellung 2 testen: zeitliche Muster nach Fiskalmonat, Wochentag und Tageszeit
- beurteilen welche Queries und Charts für die finale Analyse sinnvoll sind

Die finalen Analysen werden in `src/4_Analysis` auf dem vollständigen Datensatz ausgeführt.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc

spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_Prototype") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.executor.cores", "4") \
    .config("spark.executor.memory", "15g") \
    .config("spark.cores.max", "12") \
    .getOrCreate()

spark


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/30 16:08:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/30 16:08:56 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned_v5"

df = spark.read.parquet(processed_path)

# Nur vollständige Fiskaljahre verwenden
df = df.filter(col("is_complete_fy") == True)

df.groupBy("fy").count().orderBy("fy").show()

+----+--------+
|  fy|   count|
+----+--------+
|2023|17246732|
|2024|16162180|
|2025|16251490|
+----+--------+



In [3]:
sample_df = df.sample(fraction=0.01, seed=42)

sample_count = sample_df.count()
sample_count

498019

In [4]:
sample_df.groupBy("violation_code", "violation_description_official") \
    .count() \
    .orderBy(desc("count")) \
    .show(20, truncate=False)


[Stage 7:=============================>                             (4 + 4) / 8]

+--------------+------------------------------+------+
|violation_code|violation_description_official|count |
+--------------+------------------------------+------+
|36            |PHTO SCHOOL ZN SPEED VIOLATION|168127|
|21            |NO PARKING-STREET CLEANING    |58446 |
|38            |FAIL TO DSPLY MUNI METER RECPT|35382 |
|14            |NO STANDING-DAY/TIME LIMITS   |24783 |
|5             |BUS LANE VIOLATION            |21142 |
|7             |FAILURE TO STOP AT RED LIGHT  |20134 |
|40            |FIRE HYDRANT                  |20070 |
|20            |NO PARKING-DAY/TIME LIMITS    |18927 |
|71            |INSP. STICKER-EXPIRED/MISSING |17506 |
|70            |REG. STICKER-EXPIRED/MISSING  |11942 |
|46            |DOUBLE PARKING                |10261 |
|37            |EXPIRED MUNI METER            |8118  |
|31            |NO STANDING-COMM METER ZONE   |8044  |
|19            |NO STANDING-BUS STOP          |7167  |
|74            |FRONT OR BACK PLATE MISSING   |7150  |
|69       

In [5]:
# Fragestellung 1: Wie verändern sich die Violation Codes über die Fiskaljahre?
sample_df.groupBy("fy", "violation_code", "violation_description_official") \
    .count() \
    .orderBy("fy", desc("count")) \
    .show(30, truncate=False)

[Stage 10:=============================>                            (4 + 4) / 8]

+----+--------------+------------------------------+-----+
|fy  |violation_code|violation_description_official|count|
+----+--------------+------------------------------+-----+
|2023|36            |PHTO SCHOOL ZN SPEED VIOLATION|64126|
|2023|21            |NO PARKING-STREET CLEANING    |20593|
|2023|38            |FAIL TO DSPLY MUNI METER RECPT|11433|
|2023|14            |NO STANDING-DAY/TIME LIMITS   |8421 |
|2023|40            |FIRE HYDRANT                  |6957 |
|2023|20            |NO PARKING-DAY/TIME LIMITS    |6870 |
|2023|7             |FAILURE TO STOP AT RED LIGHT  |6667 |
|2023|71            |INSP. STICKER-EXPIRED/MISSING |6556 |
|2023|5             |BUS LANE VIOLATION            |6095 |
|2023|70            |REG. STICKER-EXPIRED/MISSING  |4478 |
|2023|46            |DOUBLE PARKING                |3596 |
|2023|37            |EXPIRED MUNI METER            |2834 |
|2023|19            |NO STANDING-BUS STOP          |2655 |
|2023|74            |FRONT OR BACK PLATE MISSING   |2644

In [6]:
# Fragestellung 2: Zeitliche Muster nach Fiskalmonat
from pyspark.sql.functions import desc

sample_df.groupBy("fy", "fm") \
    .count() \
    .orderBy("fy", "fm") \
    .show(50)

+----+---+-----+
|  fy| fm|count|
+----+---+-----+
|2023|  1|13406|
|2023|  2|17962|
|2023|  3|14921|
|2023|  4|15246|
|2023|  5|14663|
|2023|  6|12632|
|2023|  7|13525|
|2023|  8|12921|
|2023|  9|15197|
|2023| 10|14048|
|2023| 11|15212|
|2023| 12|13538|
|2024|  1|14990|
|2024|  2|14874|
|2024|  3|12625|
|2024|  4|14231|
|2024|  5|13711|
|2024|  6|11813|
|2024|  7|12241|
|2024|  8|12522|
|2024|  9|13386|
|2024| 10|13174|
|2024| 11|14212|
|2024| 12|13963|
|2025|  1|14549|
|2025|  2|14687|
|2025|  3|13937|
|2025|  4|15108|
|2025|  5|14253|
|2025|  6|12350|
|2025|  7|12017|
|2025|  8|12041|
|2025|  9|14422|
|2025| 10|14963|
|2025| 11|14461|
|2025| 12|10218|
+----+---+-----+



In [7]:
# Fragestellung 2: Zeitliche Muster nach Wochentag
sample_df.groupBy("issue_weekday") \
    .count() \
    .orderBy("issue_weekday") \
    .show()

+-------------+-----+
|issue_weekday|count|
+-------------+-----+
|            1|48831|
|            2|72017|
|            3|79933|
|            4|74434|
|            5|81299|
|            6|78808|
|            7|62697|
+-------------+-----+



In [8]:
# Fragestellung 2: Zeitliche Muster nach Tageszeit
sample_df.filter(col("violation_hour").isNotNull()) \
    .groupBy("violation_hour") \
    .count() \
    .orderBy("violation_hour") \
    .show(24)

[Stage 19:=============================>                            (4 + 4) / 8]

+--------------+-----+
|violation_hour|count|
+--------------+-----+
|             0| 6349|
|             1| 7299|
|             2| 5867|
|             3| 4728|
|             4| 4360|
|             5| 7007|
|             6|15136|
|             7|27654|
|             8|42229|
|             9|43784|
|            10|35847|
|            11|43602|
|            12|40981|
|            13|39190|
|            14|35126|
|            15|29921|
|            16|23797|
|            17|20517|
|            18|15085|
|            19|11125|
|            20|11023|
|            21| 9837|
|            22| 8554|
|            23| 7850|
+--------------+-----+



## Erkenntnisse aus dem Prototyping

Das 1%-Sample enthält 498'019 Zeilen (nach `is_complete_fy` Filter) und ist damit gross genug, um Analyseideen zu testen.

Getestete Analysefragen:

1. **Welche Violation Codes kommen am häufigsten vor, und wie verändern sie sich über die Fiskaljahre?**  
   `violation_code = 36` (PHTO SCHOOL ZN SPEED VIOLATION) ist in allen drei Fiskaljahren mit Abstand am häufigsten. Die Rangfolge der Top-Codes bleibt über die Fiskaljahre stabil. Für die finale Analyse wird `violation_description_official` verwendet.

2. **Gibt es zeitliche Muster nach Fiskalmonat?**  
   Die Monatsverteilung ist über alle drei Fiskaljahre gleichmässig (`fm=1` = Juli, `fm=12` = Juni). FY2023 zeigt leicht erhöhte Werte in `fm=2` (August) — vermutlich ein saisonales Muster.

3. **Gibt es zeitliche Muster nach Wochentag?**  
   Sonntag (`issue_weekday=1`) weist mit ~49'000 deutlich weniger Verstösse auf als die Werktage (~72'000–81'000). Samstag (`issue_weekday=7`) liegt mit ~63'000 ebenfalls tiefer. Spark kodiert Wochentage als 1=Sonntag bis 7=Samstag.

4. **Gibt es zeitliche Muster nach Tageszeit?**  
   Im Sample zeigen sich hohe Werte insbesondere zwischen 08:00 und 14:00 Uhr (Peak bei 09:00 mit ~44'000). Für diese Analyse werden nur Datensätze mit `violation_hour IS NOT NULL` verwendet.

Die getesteten Analysen werden im nächsten Schritt in `src/4_Analysis` auf dem vollständigen Datensatz ausgeführt.

In [9]:
spark.stop()